In [ ]:
# Moved to other_experiments/: run from the FLIP repo root so relative paths (data/, experiments/,
# noise_generator/, ...) and `import modules...` keep working. Safe to re-run.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_experiment.py').is_file())
os.chdir(_root)
for _p in (str(_root), str(_root / 'modules')):
    if _p not in sys.path: sys.path.insert(0, _p)

# Steganography v2: one deer (K=1) vs. a random deer from a pool (K=20)

Compares the trigger produced by two deer-in-truck encoders trained against the same surrogate ensemble:

* **K=1**: the encoder was trained with a single target deer, and every truck is blended with that deer.
* **K=20**: the encoder was trained with the top-20 deer (by worst-case surrogate margin), and each truck is blended with a deer drawn **at random** from that pool. You can also force a specific deer with `DEER_CHOICE`.

Both deer pools are read from the encoder checkpoints, exactly as `SteganographicPoisonerv2` does at poisoning time.

In [ ]:
import json, os, sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

ROOT = Path.cwd()
while not (ROOT / "run_full_flip.py").exists():      # run from FLIP/ or a subfolder
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from modules.Steganography.deer_blend import DeerBlendEncoder
from modules.Steganography.surrogates import build_surrogate
from modules.Steganography.encoder_training import quantize_st

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
SOURCE, TARGET = 9, 4

## Settings

In [ ]:
ARCH = "resnet18"          # resnet18 | resnet32 | efficientnet
ENS = 5                    # 1 | 5
N_TRUCKS = 8               # test trucks shown side by side
DEER_CHOICE = None         # None = random deer from the K=20 pool per truck; or an int in [0, 20)
SEED = 0

ENC_DIR = Path("noise_generator/encoders")
SURR_DIR = Path("noise_generator/surrogates")

## Load both encoders, their deer pools and the surrogates

In [ ]:
def load_encoder(path):
    ckpt = torch.load(path, map_location="cpu")
    enc = DeerBlendEncoder(eps=ckpt["eps"], mask_tau=ckpt.get("mask_tau", 0.08),
                           mask_floor=ckpt.get("mask_floor", 0.0), blur_sigma=ckpt.get("blur_sigma", 0.0),
                           tanh_temp=ckpt.get("tanh_temp", 1.0), use_mask=ckpt.get("use_mask", False))
    enc.load_state_dict(ckpt["state_dict"]); enc.eval().to(device)
    return enc, ckpt["deer_bank"].float().to(device), ckpt["deer_idx"], ckpt["eps"]

enc = {}
for k in (1, 20):
    path = ENC_DIR / f"{ARCH}_ens{ENS}_k{k}.pth"
    enc[k] = load_encoder(path)
    summ = json.load(open(path.with_suffix(".json")))
    print(f"K={k:2d} | {path.name} | train-set deer idx {enc[k][2][:5]}{'...' if k > 5 else ''} | "
          f"surrogate deer rate {summ['mean_surrogate_deer_rate']:.3f} | fresh D acc "
          f"{summ['fresh_discriminator_acc']:.3f} | PSNR {summ['psnr']:.1f}")

eps = enc[1][3]
assert enc[20][3] == eps, "the two encoders use different eps"
print("K=1 deer is in the K=20 pool:", enc[1][2][0] in enc[20][2])

surrogates = []
for s in range(ENS):
    m = build_surrogate(ARCH).to(device)
    m.load_state_dict(torch.load(SURR_DIR / f"{ARCH}_seed{s}.pth", map_location=device)); m.eval()
    surrogates.append(m)

## Test trucks

In [ ]:
NORM = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
test = datasets.CIFAR10("data/data_cifar10", train=False, download=False, transform=NORM)
truck_pos = [i for i, y in enumerate(test.targets) if y == SOURCE]
trucks_all = torch.stack([test[i][0] for i in truck_pos]).to(device)        # [1000, 3, 32, 32], [-1, 1]

g = torch.Generator().manual_seed(SEED)
show = torch.randperm(len(trucks_all), generator=g)[:N_TRUCKS]
trucks = trucks_all[show]

def to_img(x):           # [-1, 1] -> HWC [0, 1]
    return (x.detach().cpu().clamp(-1, 1) * 0.5 + 0.5).permute(1, 2, 0).numpy()

def noise_img(d):        # delta in [-eps, eps] -> HWC [0, 1] (amplified to full range)
    return (d.detach().cpu() / (2 * eps) + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

@torch.no_grad()
def poison(k, x, deer_ids):
    e, bank, _, _ = enc[k]
    x_p = quantize_st(e(x, bank[deer_ids]))
    return x_p, x_p - x

## The K=20 deer pool (the K=1 deer is outlined in red)

In [ ]:
bank20, idx20 = enc[20][1], enc[20][2]
fig, ax = plt.subplots(2, 10, figsize=(15, 3.4))
for j, a in enumerate(ax.flat):
    a.imshow(to_img(bank20[j])); a.set_title(f"#{j} ({idx20[j]})", fontsize=8); a.axis("off")
    if idx20[j] == enc[1][2][0]:
        for s in a.spines.values():
            s.set_visible(True); s.set_color("red"); s.set_linewidth(3)
        a.axis("on"); a.set_xticks([]); a.set_yticks([])
plt.suptitle("K=20 target-deer pool (index in pool, index in train set)"); plt.tight_layout(); plt.show()

## Side by side: one deer vs. a random deer

Rows: clean truck, then for each encoder the deer used, the poisoned truck and the noise (δ amplified from [−ε, ε] to the full range, mid-grey = 0). The title of each poisoned image gives the mean surrogate probability of *deer*.

In [ ]:
rng = np.random.default_rng(SEED)
ids1 = torch.zeros(N_TRUCKS, dtype=torch.long, device=device)
if DEER_CHOICE is None:
    ids20 = torch.as_tensor(rng.integers(0, bank20.size(0), N_TRUCKS), device=device)
else:
    ids20 = torch.full((N_TRUCKS,), int(DEER_CHOICE), device=device)

@torch.no_grad()
def p_deer(x):
    return torch.stack([torch.softmax(m(x), 1)[:, TARGET] for m in surrogates]).mean(0)

xp1, d1 = poison(1, trucks, ids1)
xp20, d20 = poison(20, trucks, ids20)
pd1, pd20, pd0 = p_deer(xp1), p_deer(xp20), p_deer(trucks)

rows = [("clean truck", [to_img(t) for t in trucks], [f"p(deer)={p:.2f}" for p in pd0]),
        ("K=1 deer", [to_img(enc[1][1][0])] * N_TRUCKS, [""] * N_TRUCKS),
        ("K=1 poisoned", [to_img(x) for x in xp1], [f"p(deer)={p:.2f}" for p in pd1]),
        ("K=1 noise", [noise_img(d) for d in d1], [f"Linf={d.abs().max()*127.5:.1f}" for d in d1]),
        ("K=20 deer", [to_img(bank20[i]) for i in ids20], [f"pool #{i}" for i in ids20.tolist()]),
        ("K=20 poisoned", [to_img(x) for x in xp20], [f"p(deer)={p:.2f}" for p in pd20]),
        ("K=20 noise", [noise_img(d) for d in d20], [f"Linf={d.abs().max()*127.5:.1f}" for d in d20])]

fig, ax = plt.subplots(len(rows), N_TRUCKS, figsize=(2 * N_TRUCKS, 2.1 * len(rows)))
for r, (label, ims, titles) in enumerate(rows):
    for c in range(N_TRUCKS):
        ax[r, c].imshow(ims[c]); ax[r, c].set_title(titles[c], fontsize=8)
        ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
    ax[r, 0].set_ylabel(label, fontsize=10)
plt.tight_layout(); plt.show()

## Does the K=20 noise actually depend on the deer?

Fix one truck and blend it with **every** deer of the pool. If the encoder learned a deer-specific code, the 20 noise patterns differ; if it collapsed to a universal trigger, they are nearly identical (pairwise correlation close to 1).

In [ ]:
TRUCK = 0                                  # index into the trucks shown above
x = trucks[TRUCK:TRUCK + 1].expand(bank20.size(0), -1, -1, -1)
xp_all, d_all = poison(20, x, torch.arange(bank20.size(0), device=device))
pd_all = p_deer(xp_all)

fig, ax = plt.subplots(2, 10, figsize=(15, 3.6))
for j, a in enumerate(ax.flat):
    a.imshow(noise_img(d_all[j])); a.set_title(f"#{j} p={pd_all[j]:.2f}", fontsize=8); a.axis("off")
plt.suptitle(f"noise on test truck {truck_pos[show[TRUCK]]} for each of the 20 deer"); plt.tight_layout(); plt.show()

flat = d_all.flatten(1)
flat = (flat - flat.mean(1, keepdim=True)) / flat.std(1, keepdim=True)
corr = (flat @ flat.T / flat.size(1)).cpu().numpy()
off = corr[~np.eye(len(corr), dtype=bool)]
plt.figure(figsize=(4.5, 3.8)); plt.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r"); plt.colorbar()
plt.title(f"noise correlation across deer\nmean off-diagonal = {off.mean():.2f}"); plt.show()

# same question across trucks: is the noise truck-specific? (K=1 vs K=20, same deer #0 / K=1 deer)
def mean_offdiag_corr(d):
    f = d.flatten(1); f = (f - f.mean(1, keepdim=True)) / f.std(1, keepdim=True)
    c = (f @ f.T / f.size(1)).cpu().numpy()
    return c[~np.eye(len(c), dtype=bool)].mean()
print(f"mean noise correlation across the {N_TRUCKS} trucks: K=1 {mean_offdiag_corr(d1):.2f} | "
      f"K=20 (random deer) {mean_offdiag_corr(d20):.2f}")

## Statistics on all 1000 test trucks

In [ ]:
@torch.no_grad()
def stats(k, random_deer=True, bs=250):
    bank = enc[k][1]
    rates, psnr, linf, energy = [], [], 0.0, []
    gen = torch.Generator(device="cpu").manual_seed(SEED)
    for b in range(0, len(trucks_all), bs):
        x = trucks_all[b:b + bs]
        ids = torch.randint(0, bank.size(0), (len(x),), generator=gen).to(device) if random_deer \
              else torch.zeros(len(x), dtype=torch.long, device=device)
        xp, d = poison(k, x, ids)
        rates.append(torch.stack([(m(xp).argmax(1) == TARGET).float() for m in surrogates]).mean(0))
        mse = ((d / 2) ** 2).mean((1, 2, 3))
        psnr.append(10 * torch.log10(1 / mse)); linf = max(linf, d.abs().max().item() * 127.5)
        energy.append(torch.fft.fft2(d.mean(1)).abs().pow(2).mean(0).cpu())
    spec = torch.stack(energy).mean(0)
    return dict(deer_rate=torch.cat(rates).mean().item(), psnr=torch.cat(psnr).mean().item(), linf=linf), spec

s1, spec1 = stats(1, random_deer=False)
s20, spec20 = stats(20, random_deer=True)
s20f, _ = stats(20, random_deer=False)
print(f"{'':28s} {'surrogate deer rate':>20s} {'PSNR':>7s} {'Linf/255':>9s}")
for name, s in [("K=1  (its deer)", s1), ("K=20 (random deer)", s20), ("K=20 (always pool #0)", s20f)]:
    print(f"{name:28s} {s['deer_rate']:20.3f} {s['psnr']:7.1f} {s['linf']:9.1f}")

# Power spectrum of the noise (luminance), low frequencies at the centre
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
for a, spec, t in [(ax[0], spec1, "K=1"), (ax[1], spec20, "K=20 random deer")]:
    im = a.imshow(torch.fft.fftshift(spec).log10().numpy(), cmap="magma"); a.set_title(f"noise power spectrum, {t}")
    a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

## All encoders of the K=20 matrix (from the JSON summaries)

In [ ]:
print(f"{'encoder':28s} {'deer rate':>9s} {'train D':>8s} {'fresh D':>8s} {'PSNR':>6s} {'min':>5s}")
for k in (1, 20):
    for arch in ("resnet18", "resnet32", "efficientnet"):
        for ens in (1, 5):
            p = ENC_DIR / f"{arch}_ens{ens}_k{k}.json"
            if p.exists():
                s = json.load(open(p))
                print(f"{s['name']:28s} {s['mean_surrogate_deer_rate']:9.3f} {s['train_discriminator_acc']:8.3f} "
                      f"{s['fresh_discriminator_acc']:8.3f} {s['psnr']:6.1f} {s['train_minutes']:5.1f}")
            else:
                print(f"{arch}_ens{ens}_k{k}: not trained yet")